# Whitebox evaluation of ADK

In [ ]:
from vertexai import Client, types
import render_util
import google.auth
from google.genai import types as genai_types
httpOptions = genai_types.HttpOptions(
    retry_options=genai_types.HttpRetryOptions(
        attempts=5,           # 최대 재시도 횟수
        initial_delay=1.0,    # 첫 대기 시간
        http_status_codes=[429, 500, 502, 503, 504] # 재시도 대상 에러 코드
    )
)
_, PROJECT_ID = google.auth.default()
LOCATION = "global"
client = Client(project=PROJECT_ID, location=LOCATION)

import os
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "true"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION

In [ ]:
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService

APP_NAME = "ecommerce_agent"

# Define Agent Tools
def search_products(query: str):
    """Searches for products based on a query."""
    # Mock response for demonstration
    if "headphones" in query.lower():
        return {"products": [{"name": "Wireless Headphones", "id": "B08H8H8H8H"}]}
    else:
        return {"products": []}

def get_product_details(product_id: str):
    """Gets the details for a given product ID."""
    if product_id == "B08H8H8H8H":
        return {"details": "Noise-cancelling, 20-hour battery life."}
    else:
        return {"error": "Product not found."}

def add_to_cart(product_id: str, quantity: int):
    """Adds a specified quantity of a product to the cart."""
    return {"status": f"Added {quantity} of {product_id} to cart."}

root_agent = Agent(
    name=APP_NAME,
    model="gemini-3.5-flash-lite",
    instruction='You are an ecommerce expert',
    tools=[search_products, get_product_details, add_to_cart],
)

# 실행을 위한 ADK Runner 및 SessionStore 세팅
session_service = InMemorySessionService()
runner = Runner(
    app_name=APP_NAME,
    agent=root_agent,
    session_service=session_service
)

In [ ]:
USER_ID = "USER_0001"
SESSION_ID = "SESSION_0001"
USER_QUERY = "Show me the details for product 'B08H8H8H8H'"

if await session_service.get_session(app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID) == None:
    await session_service.create_session(app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID)
content = genai_types.Content(role='user', parts=[genai_types.Part(text=USER_QUERY)])
events = runner.run(user_id=USER_ID, session_id=SESSION_ID, new_message=content)

final_response = None
for event in events:
    if event.is_final_response():
        final_response = event.content.parts[0].text
        print(final_response)

In [ ]:
import json
from collections import defaultdict

def remove_empty_values(data):
    """재귀적으로 빈 값({}, [], "", None)을 제거합니다."""
    if isinstance(data, dict):
        res = {k: v for k, v in ((k, remove_empty_values(v)) for k, v in data.items()) 
               if res_is_valid(v)}
        return res
    if isinstance(data, list):
        return [v for item in data if res_is_valid(v := remove_empty_values(item))]
    return data

def res_is_valid(v):
    """0, False는 유효한 값으로 인정하고 빈 구조/None/빈문자열은 제외"""
    return bool(v) or v == 0 or v is False

def rename_id_to_event_id(events):
    """각 이벤트의 최상위 'id' 키를 'event_id'로 변경"""
    return [
        {("event_id" if k == "id" else k): v for k, v in e.items()} if isinstance(e, dict) else e 
        for e in events
    ]

def format_session_turns(session, agents=None):
    """ADK Session의 events를 'turns' 단위로 그룹화"""
    turns = defaultdict(lambda: {"turn_index": len(turns), "turn_id": f"turn_{len(turns)}", "events": []})
    
    for event in session.events:
        e_dict = event.model_dump(mode="json", exclude_none=True) if hasattr(event, "model_dump") else json.loads(json.dumps(event, default=str))
        inv_id = e_dict.get("invocation_id", "default_turn")
        
        # author, content 필드만 간결하게 추출
        formatted = {k: e_dict[k] for k in ("author", "content") if k in e_dict}
        turns[inv_id]["events"].append(formatted)

    return {"agents": agents, "turns": list(turns.values())}


# --- 실행 및 결과 도출 ---
session = await session_service.get_session(app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID)

# 1. session.events -> clean dict 변환
clean_events = [
    e.model_dump(mode="json", exclude_none=True) if hasattr(e, "model_dump") else json.loads(json.dumps(e, default=str))
    for e in session.events
]

# 2. 정제된 이벤트 목록
cleaned_renamed_events = rename_id_to_event_id(remove_empty_values(clean_events))

# 3. 턴 단위 정제 결과
formatted_turns = format_session_turns(session)

In [ ]:
import pandas as pd
from vertexai import types

session_inputs = types.evals.SessionInput(
    user_id=USER_ID,
    state={},
)
agent_prompts = [
    USER_QUERY
]
agent_dataset = pd.DataFrame({
    "prompt": agent_prompts,
    "session_inputs": [session_inputs] * len(agent_prompts),
    "response": [final_response],
    "intermediate_events": [cleaned_renamed_events],
    "agent_data": [formatted_turns]
})
agent_dataset

In [ ]:
STAGING_BUCKET = f"gs://{PROJECT_ID}-agent-engine"
!gcloud storage buckets create {STAGING_BUCKET} --location=us-central1

In [ ]:
GCS_DEST = f"{STAGING_BUCKET}/output-path"
agent_info = types.evals.AgentInfo.load_from_agent(
    root_agent
)
eval_dataset = types.EvaluationDataset()
eval_dataset.candidate_name=APP_NAME
eval_dataset.eval_dataset_df = agent_dataset
evaluation_run = client.evals.create_evaluation_run(
    dataset=eval_dataset,
    agent_info=agent_info,
    agent=f"projects/{PROJECT_ID}/locations/us-central1/reasoningEngines/dummy-agent",
    metrics=[
        ##### Only working metrics
        types.RubricMetric.MULTI_TURN_TASK_SUCCESS,
        types.RubricMetric.MULTI_TURN_TOOL_USE_QUALITY,
        types.RubricMetric.MULTI_TURN_TRAJECTORY_QUALITY
    ],
    dest=GCS_DEST,
    config=types.CreateEvaluationRunConfig(
        http_options=httpOptions
    )
)

In [ ]:
import time
while evaluation_run.state not in {"SUCCEEDED", "FAILED", "CANCELLED"}:
    evaluation_run.show()
    evaluation_run = client.evals.get_evaluation_run(name=evaluation_run.name)
    time.sleep(10)

evaluation_run = client.evals.get_evaluation_run(
    name=evaluation_run.name, include_evaluation_items=True
)

# Display the Evaluation Run status and results
render_util.display_evaluation_result(evaluation_run.evaluation_item_results)

# Blackbox evaluation of Agent Runtime
### This tutorial requires following sample agent deployments
### https://github.com/cheeunlim/agent-engine-lab

In [ ]:
#AGENT_RUNTIME_RESOURCE_NAME follows below scheme
projects/{PROJECT_ID}/locations/{LOCATION}/reasoningEngines/{ENGINE_ID}

In [ ]:
#Prepare sample data
import pandas as pd
from vertexai._genai import types

session_inputs = types.evals.SessionInput(
    user_id="Test user",
    state={},
)
agent_prompts = [
    "치킨 커리 레시피가 궁금해요",
    "1주일 채식주의자를 위한 식단을 알려주세요",
]
agent_dataset = pd.DataFrame({
    "prompt": agent_prompts,
    "session_inputs": [session_inputs] * len(agent_prompts),
})
agent_dataset

In [ ]:
#Get inference result
eval_dataset = client.evals.run_inference(
    agent=AGENT_RUNTIME_RESOURCE_NAME,
    src=agent_dataset,
)

render_util.display_evaluation_dataset(eval_dataset)

In [ ]:
eval_result = client.evals.evaluate(
        dataset=eval_dataset,
        metrics=[
            types.RubricMetric.GENERAL_QUALITY,
        ],
        config=types.EvaluateMethodConfig(
            http_options=httpOptions
        )
)
render_util.display_evaluation_result(eval_result)